# Modelado — cribado individual de fracciones sociodemográficas

Este notebook llama al módulo reutilizable `src.modeling.ablacion_fracciones`. Añade una de las 15 fracciones a la vez a la misma base de seis predictores. Compara el valor fijo de 2017 y la serie anual reconstruida en h=2 y h=4, con regresión de casos y clasificación de brote. La selección usa solo validación 2021–2023. Las fracciones anuales de 2018–2024 incorporan información del extremo 2025, por lo que sus métricas son retrospectivas.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_fracciones import ejecutar_ablacion_fracciones

reporte = ejecutar_ablacion_fracciones()

h=2: 30160 filas comunes; 31 variantes


  Selección h=2: {'fija': {'regresion': 'fija__fraccion_rural', 'clasificacion': 'fija__fraccion_sin_saneamiento'}, 'anual': {'regresion': 'anual__fraccion_rural', 'clasificacion': 'anual__fraccion_hogares_lena'}}


h=4: 30030 filas comunes; 31 variantes


  Selección h=4: {'fija': {'regresion': 'fija__fraccion_agua_red', 'clasificacion': 'fija__fraccion_sin_seguro'}, 'anual': {'regresion': 'anual__fraccion_hogares_celular', 'clasificacion': 'anual__fraccion_sin_saneamiento'}}


In [2]:
import pandas as pd

filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        folds = resultado['folds_validacion']
        filas.append({'h': int(h), 'variante': variante,
                      'F1_regresion': sum(f['regresion']['alerta_derivada']['f1'] for f in folds) / len(folds),
                      'F1_clasificacion': sum(f['clasificacion']['alerta_directa']['f1'] for f in folds) / len(folds)})
ranking = pd.DataFrame(filas).sort_values(['h', 'variante'])
display(ranking)
for h, bloque in reporte['horizontes'].items():
    print('h =', h, 'selección:', bloque['elegidas_solo_validacion'])
    print('Ganadoras por temporada:', bloque['ganadora_por_temporada'])

,h,variante,F1_regresion,F1_clasificacion
24,2,anual__fraccion_agua_cisterna,0.672526,0.601583
23,2,anual__fraccion_agua_red,0.673160,0.600835
27,2,anual__fraccion_alumbrado_red,0.673757,0.622533
20,2,anual__fraccion_analfabeta_15_mas,0.678919,0.595537
25,2,anual__fraccion_desague_red,0.675381,0.590344
...,...,...,...,...
37,4,fija__fraccion_pared_precaria,0.559010,0.551811
38,4,fija__fraccion_piso_tierra,0.563365,0.559691
32,4,fija__fraccion_rural,0.570060,0.535347
42,4,fija__fraccion_sin_saneamiento,0.563942,0.582250


h = 2 selección: {'fija': {'regresion': 'fija__fraccion_rural', 'clasificacion': 'fija__fraccion_sin_saneamiento'}, 'anual': {'regresion': 'anual__fraccion_rural', 'clasificacion': 'anual__fraccion_hogares_lena'}}
Ganadoras por temporada: {'fija': {'regresion': {'temporada_2021': 'fija__fraccion_agua_cisterna', 'temporada_2022': 'fija__fraccion_hogares_celular', 'temporada_2023': 'base_6'}, 'clasificacion': {'temporada_2021': 'fija__fraccion_sin_saneamiento', 'temporada_2022': 'fija__fraccion_sin_seguro', 'temporada_2023': 'fija__fraccion_sin_seguro'}}, 'anual': {'regresion': {'temporada_2021': 'anual__fraccion_rural', 'temporada_2022': 'anual__fraccion_hogares_refrigeradora', 'temporada_2023': 'base_6'}, 'clasificacion': {'temporada_2021': 'anual__fraccion_hogares_lena', 'temporada_2022': 'anual__fraccion_sin_saneamiento', 'temporada_2023': 'anual__fraccion_sin_seguro'}}}
h = 4 selección: {'fija': {'regresion': 'fija__fraccion_agua_red', 'clasificacion': 'fija__fraccion_sin_seguro'}, 

In [3]:
filas = []
for h, bloque in reporte['horizontes'].items():
    for variante, resultado in bloque['variantes'].items():
        for fold in resultado['folds_prueba']:
            for enfoque, metrica in (('regresion', fold['regresion']['alerta_derivada']),
                                     ('clasificacion', fold['clasificacion']['alerta_directa'])):
                filas.append({'h': int(h), 'bloque': fold['bloque'], 'variante': variante,
                              'enfoque': enfoque, 'positivos': metrica['positivos'],
                              'VP': metrica['vp'], 'FP': metrica['fp'], 'F1': metrica['f1']})
display(pd.DataFrame(filas).sort_values(['h', 'bloque', 'enfoque', 'variante']))

,h,bloque,variante,enfoque,positivos,VP,FP,F1
19,2,calendario_2025,anual__fraccion_hogares_lena,clasificacion,3,1,258,0.007634
15,2,calendario_2025,anual__fraccion_rural,clasificacion,3,2,260,0.015094
3,2,calendario_2025,base_6,clasificacion,3,2,287,0.013699
7,2,calendario_2025,fija__fraccion_rural,clasificacion,3,2,254,0.015444
11,2,calendario_2025,fija__fraccion_sin_saneamiento,clasificacion,3,1,207,0.009479
18,2,calendario_2025,anual__fraccion_hogares_lena,regresion,3,0,3,0.000000
14,2,calendario_2025,anual__fraccion_rural,regresion,3,0,3,0.000000
2,2,calendario_2025,base_6,regresion,3,0,3,0.000000
6,2,calendario_2025,fija__fraccion_rural,regresion,3,0,3,0.000000
10,2,calendario_2025,fija__fraccion_sin_saneamiento,regresion,3,0,3,0.000000


La interpretación y los límites están en `docs/modeling/ablacion_fracciones.md`. Los bloques 2024 y 2025 se consultaron solo después de fijar las variantes. La selección entre 15 fracciones con tres temporadas es exploratoria.